Loading Data

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.sql("""
    CREATE VIEW matches AS
    SELECT * FROM read_csv_auto('../data/processed/matches.csv', types={'Season': 'VARCHAR'})
""")
con.sql("""
    CREATE VIEW team_form AS
    SELECT * FROM read_csv_auto('../data/processed/team_form.csv', types={'Season': 'VARCHAR'})
""")

Joining home and away features

In [2]:
con.sql("""
    CREATE OR REPLACE VIEW model_table AS
    SELECT m.Season, m.Date, m.HomeTeam, m.AwayTeam,
           m.FTHG AS home_goals, m.FTAG AS away_goals, m.FTR,
           m.B365H, m.B365D, m.B365A,

           h.prior_matches        AS h_prior,
           h.points_last5         AS h_points_last5,
           h.goals_for_last5      AS h_gf5,
           h.goals_against_last5  AS h_ga5,
           h.goals_for_last10     AS h_gf10,
           h.goals_against_last10 AS h_ga10,
           h.days_rest            AS h_rest,

           a.prior_matches        AS a_prior,
           a.points_last5         AS a_points_last5,
           a.goals_for_last5      AS a_gf5,
           a.goals_against_last5  AS a_ga5,
           a.goals_for_last10     AS a_gf10,
           a.goals_against_last10 AS a_ga10,
           a.days_rest            AS a_rest
    FROM matches m
    JOIN team_form h ON h.Team = m.HomeTeam AND h.Date = m.Date AND h.Venue = 'H'
    JOIN team_form a ON a.Team = m.AwayTeam AND a.Date = m.Date AND a.Venue = 'A'
""")

Checks

In [3]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM model_table) AS model_rows,
           (SELECT COUNT(*) FROM matches) AS match_rows
""").df()

,model_rows,match_rows
0,3520,3520


In [4]:
df = con.sql("SELECT * FROM model_table ORDER BY Date").df()
row = df[df["HomeTeam"] == "Arsenal"].iloc[20]
print(row[["Date", "HomeTeam", "AwayTeam", "h_points_last5", "h_gf5"]])

con.sql(f"""
    SELECT Date, Team, points_last5, goals_for_last5
    FROM team_form
    WHERE Team = 'Arsenal' AND Date = '{row['Date'].date()}'
""").df()

Date              2018-08-25 00:00:00
HomeTeam                      Arsenal
AwayTeam                     West Ham
h_points_last5                      6
h_gf5                             1.8
Name: 400, dtype: object


,Date,Team,points_last5,goals_for_last5
0,2018-08-25,Arsenal,6,1.8


Filter and split

In [5]:
n_before = len(df)
df = df[(df["h_prior"] >= 5) & (df["a_prior"] >= 5)].reset_index(drop=True)
print(n_before, "->", len(df), "rows")


3520 -> 3401 rows


In [6]:
train = df[df["Season"] <= "2324"]
valid = df[df["Season"] == "2425"]
test  = df[df["Season"] == "2526"]

print(len(train), len(valid), len(test))

2560 375 375


In [7]:
df.to_csv("../data/processed/model_table.csv", index=False)

## Summary: the training table

**What this notebook built**
A one-row-per-match table (`model_table`) for predicting the goals each team scores. It combines match results with the rolling-form features from `03_sql`, so each match has the home team's form and the away team's form side by side.

**How it was built**
- Loaded `matches.csv` and `team_form.csv` into DuckDB
- Joined `team_form` twice (home team, away team) and prefixed the columns `h_` and `a_`
- Kept the real goals (`home_goals`, `away_goals`) as targets
- Kept the bookmaker odds for benchmarking only, never as features

**Features (all calculated before kickoff)**
- Points over the last 5 matches
- Average goals scored and conceded over the last 5 and 10 matches
- Days of rest
- `prior_matches`, used to filter out thin history

**Checks**
- `model_table` has 3520 rows and `matches` has 3520, so the joins lost and duplicated nothing: passed
- The data covers 34 distinct teams, seasons 1718 to 2627

**Filtering and split**
- Dropped matches where either team had fewer than 5 prior matches: 119 rows removed, 3401 remaining
- Time-based split with no shuffling:
  - Train: seasons up to 2023/24 (2560 matches)
  - Validation: 2024/25 (375 matches), for comparing models
  - Test: 2025/26 (375 matches), used once at the end
- The partial 2026/27 season (91 matches) is held out for a live demo

**Leakage rules followed**
- Every feature uses only matches played before the one being predicted
- No shots, fouls, cards or odds from the current match are used as features
- The test season is not used for any decision before the final evaluation

**Limitations**
- Early matches and newly promoted teams are excluded because of thin history
- Form carries across seasons, so late-season games feed early-season features
- No player, injury or lineup information is included

## Next: modeling
1. Baseline: predict the training average of home and away goals for every match
2. Poisson regression for expected home and away goals
3. Convert expected goals into scoreline and win/draw/loss probabilities
4. Evaluate once on the test season against the baseline, "always home win" and the bookmaker odds
